# Objectifs partie 1

### Cahier des charges streamlit :

#### 1ere page : accueil

- accueil : photo, blabla...

#### 2eme page : Timeline event

Upper part : Click for 1 of the 6 event types and get a time series of the number of tweets per day/weeks related to this event 

Middle part : generate a wordcloud of the most used words in the tweets related to this event (after preprocessing)

Lower part : Possibility to enter a word and get the time series of the number of tweets per day/weeks related to this word among the 6 events

#### 3eme page : 

#### 4eme page : Information Retrieval

Barre de recherche : possibilité de rentrer une recherche et renvoie les top 5 tweets les plus proches de la recherche (avec les metadata importantes)

In [1]:
import sys
import os
import pandas as pd
import json
import spacy
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from data_loader import load_pkl

In [2]:
from tqdm import tqdm
tqdm.pandas()

In [35]:
dfs = load_pkl(root_path = "/Users/hugorameil/Desktop/Code/GitHub/NLP_project/")

## Dataframes creation

In [36]:
df_tweet = (
    dfs["Tweet"]
    .assign(created_at=lambda x: pd.to_datetime(x["created_at"].str[:10]))
)

df_event = (dfs["Event"])
df_user = (dfs["User"])
df_hashtag = (dfs["Hashtag"])
df_post_category = (dfs["PostCategory"])

Type of events : ['wildfire', 'earthquake', 'flood', 'typhoon', 'shooting', 'bombing']

In [37]:
merge = (
    df_tweet
    .merge(
        df_event,
        how="inner",
        left_on="topic",
        right_on="trecisid",
        suffixes=("_tweet", "_event"),
    )
    .assign(week_start = lambda df : df["created_at"].dt.to_period("W").dt.start_time)
)
merge.head()

,isTruncated,possibly_sensitive,created_at,retweet_count,annotation_annotated,is_quote_status,annotation_num_judgements,annotation_postPriority,id_str,topic,favorite_count,id_tweet,text,id_event,eventType,trecisid,week_start
0,False,False,2012-06-09,0,True,False,3,Low,211281973870727170,TRECIS-CTIT-H-001,1,211281973870727170,#colorado. Told you its #amazing http://t.co/6...,fireColorado2012,wildfire,TRECIS-CTIT-H-001,2012-06-04
1,False,False,2012-06-09,3,True,False,1,Medium,211557401231495171,TRECIS-CTIT-H-001,0,211557401231495171,RT @northfortynews: Tanker helicopter heads up...,fireColorado2012,wildfire,TRECIS-CTIT-H-001,2012-06-04
2,False,True,2012-06-09,2,True,False,1,High,211565974422425600,TRECIS-CTIT-H-001,0,211565974422425600,#Evacuation center Cache La Poudre Middle Scho...,fireColorado2012,wildfire,TRECIS-CTIT-H-001,2012-06-04
3,False,True,2012-06-09,0,True,False,1,Medium,211607187653533697,TRECIS-CTIT-H-001,0,211607187653533697,20F degrees cooler tomorrow in North Central &...,fireColorado2012,wildfire,TRECIS-CTIT-H-001,2012-06-04
4,False,False,2012-06-10,0,True,False,1,Medium,211654415503990784,TRECIS-CTIT-H-001,0,211654415503990784,FEMA has authorized the use of federal funds t...,fireColorado2012,wildfire,TRECIS-CTIT-H-001,2012-06-04


#### store the csv event per weeks

In [38]:
id_event_week = (merge
.groupby(["week_start", "id_event"]).count()
.reset_index()
.set_index('week_start')
.assign(tweet_number = lambda x: x['id_tweet'])
.filter(['id_event', 'tweet_number'])
)

# id_event_week.to_csv('/Users/hugorameil/Library/Mobile Documents/com~apple~CloudDocs/Desktop/Code/GitHub/NLP_project/data/Nodes/id_event_week.csv')

#### same job for type of events

In [39]:
toe_week = (merge
.groupby(["week_start", "eventType"]).count()
.reset_index()
.set_index('week_start')
.assign(tweet_number = lambda x: x['id_tweet'])
.filter(['eventType', 'tweet_number'])
)

#toe_week.to_csv('/Users/hugorameil/Library/Mobile Documents/com~apple~CloudDocs/Desktop/Code/GitHub/NLP_project/data/Nodes/toe_week.csv')

# Information retrieval

In [3]:
nlp = spacy.load('en_core_web_sm')

In [11]:
def preprocess_text(text):
    # Process the text with spaCy NLP pipeline
    doc = nlp(text)
    
    # Tokenize, remove stopwords, punctuation, and lemmatize
    preprocessed_text = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
    
    # Join the preprocessed tokens back into a single string
    return ' '.join(preprocessed_text)

### preprocessing de df_tweet.text

In [18]:
df_tweet['cleaned_text'] = df_tweet['text'].progress_apply(preprocess_text)

100%|██████████| 55986/55986 [10:58<00:00, 85.06it/s] 


### preprocess the query in the same way

In [19]:
query = "Colorado wildfires" 

In [20]:
cleaned_query = preprocess_text(query)
cleaned_query

'colorado wildfire'

### word embeddings

In [21]:
def get_doc_vector(text):
    doc = nlp(text)
    return doc.vector

In [22]:
# Get the vector for the query
query_vector = get_doc_vector(cleaned_query)
query_vector

array([-0.25488734, -0.738684  ,  0.6436641 ,  0.05126882,  0.40231013,
       -0.1821978 ,  0.8319236 ,  0.88649744,  0.05397488, -0.47421914,
        1.0870479 , -0.07472178,  0.04397601, -0.14572711, -0.09607732,
        0.12669142, -0.2791982 , -0.0767964 ,  0.3787641 , -0.53678906,
       -1.110049  ,  0.7518661 ,  0.08033849,  0.39082658,  0.29280046,
       -0.0336172 ,  0.19156638,  0.18292774, -0.2478528 ,  0.42700863,
       -0.27696598,  0.58303404, -0.12715119,  0.11413984, -0.91244024,
       -0.58006936, -0.10741475,  0.283637  , -0.6776035 ,  0.5388466 ,
       -0.57926756,  1.0116041 , -0.7239155 ,  1.2137694 ,  0.09950759,
        0.6397027 , -0.8367624 ,  0.10258079, -0.5496845 , -0.38872504,
       -0.280263  ,  0.46640727,  0.62041384, -0.599718  ,  0.36694384,
       -0.27991188,  0.7244985 ,  0.38866752,  0.01175097,  0.03840043,
       -0.8592553 , -0.2730507 , -0.1228788 , -0.59453326, -0.41731024,
       -0.40508753, -0.23674811, -0.14277393, -0.6513762 , -0.04

small model this is why we have only 96 dimensions

In [23]:
# Get vectors for all tweets in the DataFrame
df_tweet['vector'] = df_tweet['cleaned_text'].progress_apply(get_doc_vector)

100%|██████████| 55986/55986 [08:31<00:00, 109.47it/s]


In [24]:
df_tweet_vector = (
    df_tweet
    .filter(['id','created_at', 'text', 'topic', 'vector'])
)

df_tweet_vector.head()

,id,created_at,text,topic,vector
0,211281973870727170,2012-06-09,#colorado. Told you its #amazing http://t.co/6...,TRECIS-CTIT-H-001,"[0.05943238, -0.6954139, 0.09215439, -0.076830..."
1,211557401231495171,2012-06-09,RT @northfortynews: Tanker helicopter heads up...,TRECIS-CTIT-H-001,"[0.091286846, -0.52605975, 0.3576188, 0.450296..."
2,211565974422425600,2012-06-09,#Evacuation center Cache La Poudre Middle Scho...,TRECIS-CTIT-H-001,"[0.15135013, -0.52674055, 0.20749283, 0.241616..."
3,211607187653533697,2012-06-09,20F degrees cooler tomorrow in North Central &...,TRECIS-CTIT-H-001,"[0.22207421, -0.4004284, -0.032802563, -0.1272..."
4,211654415503990784,2012-06-10,FEMA has authorized the use of federal funds t...,TRECIS-CTIT-H-001,"[0.0051706224, -0.72611046, 0.039333582, 0.312..."


In [25]:
df_tweet_vector.to_csv('/Users/hugorameil/Desktop/Code/GitHub/NLP_project/data/Nodes/tweet_vector.csv', sep=';')

In [26]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)

    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    
    return dot_product / (norm_vec1 * norm_vec2)

In [29]:
def get_top_5_tweets(df, query_vector):
    # Compute cosine similarity
    df['cosine_similarity'] = df['vector'].apply(lambda x: cosine_similarity(query_vector, x))
    
    # Sort tweets by cosine similarity
    df = df.sort_values(by='cosine_similarity', ascending=False)
    
    # Get top 5 tweets
    top_5_tweets = df.head(5)
    
    return top_5_tweets

In [30]:
get_top_5_tweets(df_tweet_vector, query_vector).filter(['id', 'text', 'cosine_similarity'])

,id,text,cosine_similarity
585,218476044112498688,These colorado wildfires are no joke,0.907789
235,217103105794375680,Colorado is on fire,0.899224
51367,217828760534265857,All of Colorado is on fire... Except for the R...,0.857781
560,218387913375883264,its a shame Colorado couldn't have all this ra...,0.845156
427,218025655571464192,Please keep Colorado and Florida in your praye...,0.838575
